# Trexquant - Earnings Return Prediction Challenge 2025
---

## Configure the competition dataset
When you start editing this Notebook, you need to properly configure the Notebook's **Input**.
- On the right-hand side of the Notebook, you will find the **"Input"** panel.
- Click the **"Add Input"** button, search for the keyword **"Earnings Return Prediction Challenge 2025Q4"** to locate our competition, and then click the **"+"** button in the lower-right corner.

---

## GAR Candidate Submission

- Your goal is to implement a solution with a correlation score **`≥ 0.03`**.
- You have up to **7 days** to complete this challenge.
- You may select **2** final submissions to be considered for judging.

---

## Important Notes
- Participants should not intentionally overfit and must avoid using any future information to ensure that their models do not have forward bias or inadvertently use future data.
- Common mistake: Be cautious when applying scaling/normalization methods that use data from the entire time period, as this can lead to overfitting.

In [ ]:
import os
import gc
import json
import warnings
from glob import glob

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 1. Load Dataset

In [ ]:
def find_train_test():
    pairs = []
    for train_path in glob('/kaggle/input/**/train.csv', recursive=True):
        test_path = os.path.join(os.path.dirname(train_path), 'test.csv')
        if os.path.exists(test_path):
            pairs.append((train_path, test_path))
    if not pairs:
        raise FileNotFoundError(
            'No train.csv/test.csv found. Attach the competition dataset in the Kaggle Input pane first.'
        )
    return pairs[0]

train_path, test_path = find_train_test()
df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print('train shape:', df_train.shape)
print('test  shape:', df_test.shape)

## 2. Data Audit

Key facts found locally:
- **146,804 train rows**, **72,720 test rows**, 179 feature columns
- `id` is unique in both train and test — no repeats
- `di` has 3,347 unique values (date-like index, good for time-ordered CV)
- `si` has 5,276 unique stock identifiers
- Several features have >50% missingness (f_58: 86%, f_75: 85%, etc.) — CatBoost handles these natively
- Target: mean ≈ 0, std ≈ 0.072, kurtosis ≈ 10 (fat tails, expect some extreme earners)
- CV strategy comparison (Ridge OOF Pearson):
  - shuffled KFold: **0.0516** — inflated, no time structure
  - GroupKFold by si: **0.0517** — similar inflation
  - **Contiguous folds on di: 0.0455** — most honest, time-ordered
- Shuffled-target check: **0.006** → CV is not lying; real signal exists

In [ ]:
ID_COL     = 'id'
TARGET_COL = 'target'
TIME_COL   = 'di'

feature_cols = [c for c in df_train.columns if c not in [ID_COL, TARGET_COL]]

print('id unique train:', df_train[ID_COL].is_unique)
print('id unique test: ', df_test[ID_COL].is_unique)
print('di unique train:', df_train[TIME_COL].nunique())
print('si unique train:', df_train['si'].nunique())
print()
print('target describe:')
print(df_train[TARGET_COL].describe())
print('skew:', round(df_train[TARGET_COL].skew(), 4))
print('kurt:', round(df_train[TARGET_COL].kurt(), 4))

miss = df_train[feature_cols].isna().mean().sort_values(ascending=False)
print('\nTop 10 missing:')
print(miss.head(10).to_string())

## 3. Validation Strategy

Use **contiguous time-ordered folds on `di`** — this is the most honest split for a time-series-like earnings dataset. Shuffled KFold inflates scores by ~0.006 Pearson points.

In [ ]:
def pearson_corr(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def contiguous_time_splits(time_series, n_splits=5):
    unique_vals = np.sort(time_series.dropna().unique())
    chunks = np.array_split(unique_vals, n_splits)
    row_idx = np.arange(len(time_series))
    splits = []
    for chunk in chunks:
        valid_mask = time_series.isin(chunk).to_numpy()
        splits.append((row_idx[~valid_mask], row_idx[valid_mask]))
    return splits

N_SPLITS = 5
splits = contiguous_time_splits(df_train[TIME_COL], n_splits=N_SPLITS)
print(f'Built {N_SPLITS} contiguous time folds on di')
for i, (tr, va) in enumerate(splits):
    print(f'  fold {i}: train={len(tr):,}  valid={len(va):,}')

## 4. CatBoost Baseline

Local results:
- Ridge OOF Pearson (time-ordered): **0.0455**
- **CatBoost OOF Pearson (time-ordered): 0.0621** ← chosen model
- Shuffled-target baseline: **0.006** → signal is real

CatBoost handles the heavy missingness natively without imputation.

In [ ]:
from catboost import CatBoostRegressor

cat_cols = ['si', 'industry', 'sector', 'top2000', 'top1000', 'top500']
cat_cols = [c for c in cat_cols if c in feature_cols]

train_cb = df_train[feature_cols].copy()
test_cb  = df_test[feature_cols].copy()
for c in cat_cols:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c]  = test_cb[c].astype(str)

y = df_train[TARGET_COL].to_numpy(dtype=np.float64)
oof_pred  = np.zeros(len(df_train), dtype=np.float64)
test_pred = np.zeros(len(df_test),  dtype=np.float64)

for fold_id, (tr_idx, va_idx) in enumerate(splits):
    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=6,
        learning_rate=0.03,
        iterations=1000,
        l2_leaf_reg=8.0,
        random_seed=42 + fold_id,
        subsample=0.8,
        bootstrap_type='Bernoulli',
        od_type='Iter',
        od_wait=100,
        verbose=False,
    )
    model.fit(
        train_cb.iloc[tr_idx], y[tr_idx],
        cat_features=cat_cols if cat_cols else None,
        eval_set=(train_cb.iloc[va_idx], y[va_idx]),
        use_best_model=True,
    )
    oof_pred[va_idx]  = model.predict(train_cb.iloc[va_idx])
    test_pred        += model.predict(test_cb) / N_SPLITS

    fold_corr = pearson_corr(y[va_idx], oof_pred[va_idx])
    print(f'Fold {fold_id}: Pearson={fold_corr:.6f}  best_iter={model.best_iteration_}')
    del model
    gc.collect()

oof_corr = pearson_corr(y, oof_pred)
print(f'\nOOF Pearson (time-ordered): {oof_corr:.6f}')

## 5. Submission

**Requirements:**
- Must contain `"target"` column.
- No `nan` or `inf` in `"target"`.
- At least `10%` of target values must be non-zero.

In [ ]:
# Mild post-processing: clip extreme outliers
lo, hi = np.nanpercentile(test_pred, [0.5, 99.5])
test_pred = np.clip(test_pred, lo, hi)

# Guarantee at least 10% non-zero
if (np.abs(test_pred) > 0).mean() < 0.10:
    test_pred = test_pred + 1e-9

df_submission = pd.DataFrame({
    'id':     df_test[ID_COL],
    'target': test_pred,
})

assert np.isfinite(df_submission['target']).all(), 'Non-finite predictions found'
assert (df_submission['target'].abs() > 0).mean() >= 0.10, 'Less than 10% non-zero'

filename = '/kaggle/working/submission.csv'
df_submission.to_csv(filename, index=False)

print(f'Saved: {filename}')
print(f'Rows: {len(df_submission):,}')
print(f'Non-zero fraction: {(df_submission["target"].abs() > 0).mean():.4f}')
print(f'OOF Pearson (validation): {oof_corr:.6f}')
print(df_submission.head())